In [0]:
from uuid import uuid4
from datetime import datetime, timezone
from pyspark.sql import functions as F

BRONZE_TABLE = "cat_adblh_dev.bronze.shipments_postgres_raw"
AUDIT_TABLE = (
    "cat_adblh_dev.operations.shipments_delivery_reconciliation"
)
LANDING_ROOT = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments"
)

# PROBLEM 1 — Missing deliveries can go unnoticed.
# Start from the known inventory so a delivery remains visible in the audit
# even when it cannot be observed in ADLS or has no records in Bronze.
#
# PROBLEM 5 — Audit results can overstate certainty.
# These counts are previously observed local extraction counts,
# not producer-certified source totals.
deliveries = [
    ("snapshot_baseline", 1000),
    ("batch_cd8e5615670f442eb00a537ca2f38554", 2),
    ("batch_870d88642dfb46ab90018d4d288f93fc", 1),
    ("batch_bdc5ed4fa27743938cc9f979320eb855", 3),
    ("batch_3c44f133fa16444483654d7aa7728b28", 3),
]

# Identify this reconciliation execution.
# This is not the run ID of an ingestion attempt.
audit_run_id = str(uuid4())
audit_started_at = datetime.now(timezone.utc)
results = []

for delivery_id, reference_rows in deliveries:
    landing_path = f"{LANDING_ROOT}/delivery_id={delivery_id}/"

    # PROBLEM 5 — Audit results can overstate certainty.
    # Begin with unknown measurements. Assign zero only when observation
    # succeeds and actually establishes that the count is zero.
    landing_files = None
    landing_rows = None
    bronze_files = None
    bronze_rows = None
    errors = []

    try:
        # PROBLEM 3 — Empty deliveries leave no row metadata.
        # Count physical Parquet files separately from their records.
        # One readable file with zero rows differs from no Parquet files.
        # This inventory uses direct files, not nested directories.
                # PROBLEM — We cannot quickly locate missing records.
        # Include both file extensions observed in this project's landing data.
        #
        # PROBLEM — Audit results can overstate certainty.
        # Selection by extension does not validate the file format.
        # The subsequent Parquet read establishes readability.
        files = [
            item.path
            for item in dbutils.fs.ls(landing_path)
            if not item.isDir()
            and item.name.lower().endswith((".parquet", ".par"))
        ]
        landing_files = len(files)
        landing_rows = (
            spark.read.parquet(*files).count() if files else 0
        )

    except Exception as exc:
        # PROBLEM 5 — Audit results can overstate certainty.
        # Preserve the observation error. A missing path, access failure,
        # or unreadable file must not be reported as a measured zero.
        errors.append(
            f"LANDING: {type(exc).__name__}: {str(exc)[:500]}"
        )

    try:
        # PROBLEM 2 — We cannot quickly locate missing records.
        # Measure the Bronze side of the ADLS-to-Bronze boundary.
        # Keeping both measurements allows a side-by-side comparison.
        observed = (
            spark.table(BRONZE_TABLE)
            .filter(F.col("_delivery_id") == delivery_id)
            .agg(
                F.count("*").alias("rows"),
                F.countDistinct("_source_file").alias("files"),
            )
            .first()
        )
        bronze_rows = observed["rows"]
        bronze_files = observed["files"]

    except Exception as exc:
        # PROBLEM 5 — Audit results can overstate certainty.
        # An unsuccessful Bronze observation remains unknown.
        errors.append(
            f"BRONZE: {type(exc).__name__}: {str(exc)[:500]}"
        )

    # PROBLEM 4 — Successful execution can be mistaken for completeness.
    # Calculate a reconciliation result independently of notebook success.
    # COUNTS_MATCH establishes count agreement for this known inventory;
    # it does not certify source completeness, values, or downstream readiness.
    if errors:
        status = "UNKNOWN"
    elif (
        landing_files == 1
        and bronze_files == 1
        and landing_rows == reference_rows
        and bronze_rows == landing_rows
    ):
        status = "COUNTS_MATCH"
    else:
        status = "MISMATCH"

    # PROBLEM 1 — Missing deliveries can go unnoticed.
    # Record every reference delivery, including mismatches and unknowns.
    #
    # PROBLEM 5 — Audit results can overstate certainty.
    # Retain the reference origin alongside measurements and errors.
    results.append((
        audit_run_id,
        audit_started_at,
        datetime.now(timezone.utc),
        delivery_id,
        landing_path,
        "PREVIOUS_LOCAL_EXTRACTION_OBSERVATION",
        1,
        reference_rows,
        landing_files,
        landing_rows,
        bronze_files,
        bronze_rows,
        status,
        " | ".join(errors) if errors else None,
    ))

schema = """
    audit_run_id STRING,
    audit_started_at TIMESTAMP,
    observed_at TIMESTAMP,
    delivery_id STRING,
    landing_path STRING,
    reference_origin STRING,
    reference_files BIGINT,
    reference_rows BIGINT,
    landing_files BIGINT,
    landing_rows BIGINT,
    bronze_files BIGINT,
    bronze_rows BIGINT,
    reconciliation_status STRING,
    observation_error STRING
"""

audit_df = spark.createDataFrame(results, schema=schema)

# Persist this audit snapshot, including MISMATCH and UNKNOWN results.
# Rerunning adds another snapshot; it does not change Bronze shipment data.
audit_df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

# PROBLEM 2 — We cannot quickly locate missing records.
# Show reference, landing, and Bronze counts together.
#
# PROBLEM 4 — Successful execution can be mistaken for completeness.
# Show the reconciliation status even when the notebook finishes successfully.
#
# One consolidated check: display the persisted results from this execution.
display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("audit_run_id") == audit_run_id)
    .select(
        "delivery_id",
        "reference_rows",
        "landing_files",
        "landing_rows",
        "bronze_files",
        "bronze_rows",
        "reconciliation_status",
        "observation_error",
    )
    .orderBy("delivery_id")
)